# Tahap 08: Pengujian Eksperimen Model (Baseline & Ablation)

Tahap ini dirancang untuk melakukan eksperimen komparatif guna memvalidasi efektivitas model utama. Terdapat dua eksperimen utama:
1. **Ablation Study:** Menguji performa algoritma *Logistic Regression* menggunakan fitur TF-IDF secara utuh (tanpa seleksi fitur *Chi-Square*) untuk mengetahui dampak reduksi dimensi terhadap metrik akurasi.
2. **Baseline Model:** Melatih algoritma *Naive Bayes* (*MultinomialNB*) menggunakan fitur hasil seleksi *Chi-Square* sebagai tolok ukur (*baseline*) performa klasifikasi untuk dibandingkan dengan model *Logistic Regression* utama.

In [ ]:
# --- LIBRARIES ---
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# --- 1. KONFIGURASI DIREKTORI & PATH ---
BASE_DIR = r'C:\Users\Asus\Desktop\SKRIPSI\1. MAIN'

# --- 2. PENYELARASAN MAPPING LABEL ---
# Memastikan mapping selaras dengan pemrosesan awal
custom_mapping = {'keluhan': 0, 'saran': 1, 'pujian': 2}
target_names = ['Keluhan (0)', 'Saran (1)', 'Pujian (2)']

print("Konfigurasi direktori dan mapping label berhasil disiapkan.")
print("="*60)

## 8.1 Eksperimen 1: *Logistic Regression* Tanpa *Chi-Square*
Menguji performa algoritma regresi logistik pada matriks TF-IDF penuh (belum direduksi) dengan tetap mempertahankan konfigurasi hyperparameter *imbalanced data* (`class_weight='balanced'`).

In [ ]:
# ==============================================================================
# EKSPERIMEN 1: LOGISTIC REGRESSION TANPA CHI-SQUARE
# ==============================================================================

print("[1/2] Memuat Data TF-IDF Murni (Tanpa Seleksi Fitur)...")
path_tfidf = os.path.join(BASE_DIR, 'tfidf_model_rev_2.0.pkl')

with open(path_tfidf, 'rb') as f:
    X_train_full, X_test_full, y_train_raw, y_test_raw = pickle.load(f)

# Menerapkan Custom Mapping pada data label murni
y_train_full = y_train_raw.map(custom_mapping)
y_test_full = y_test_raw.map(custom_mapping)

print(f"      Jumlah Dimensi Fitur: {X_train_full.shape[1]}")
print("      Sedang melatih model Logistic Regression (Konfigurasi Original)...")

# --- INISIALISASI & TRAINING MODEL ---
# Konfigurasi dipertahankan sama dengan model utama agar perbandingan valid
model_tanpa_chi = LogisticRegression(
    multi_class='multinomial', 
    solver='lbfgs', 
    max_iter=1000, 
    class_weight='balanced', # Menangani ketimpangan kelas
    random_state=42
)

model_tanpa_chi.fit(X_train_full, y_train_full)

# --- EVALUASI EKSPERIMEN 1 ---
y_pred_full = model_tanpa_chi.predict(X_test_full)
acc_full = accuracy_score(y_test_full, y_pred_full)

print("\nHASIL EKSPERIMEN 1 (TANPA SELEKSI FITUR CHI-SQUARE):")
print("-" * 60)
print(f"Akurasi Keseluruhan: {acc_full:.2%}")
print("\nClassification Report:")
print(classification_report(y_test_full, y_pred_full, target_names=target_names))
print("="*60)

## 8.2 Eksperimen 2: *Baseline Model* (*Naive Bayes*)
Menguji performa algoritma klasifikasi probabilistik tradisional (*Multinomial Naive Bayes*) menggunakan fitur yang telah diseleksi sebagai batas performa minimal (*baseline*) klasifikasi teks.

In [ ]:
# ==============================================================================
# EKSPERIMEN 2: BASELINE MODEL (NAIVE BAYES)
# ==============================================================================

print("\n[2/2] Memuat Data Final (Dengan Seleksi Fitur Chi-Square)...")
path_chi = os.path.join(BASE_DIR, 'selector_chi2_rev_2.0.pkl')

with open(path_chi, 'rb') as f:
    X_train_chi, X_test_chi, y_train_raw_chi, y_test_raw_chi = pickle.load(f)

# Menerapkan Custom Mapping
y_train_chi = y_train_raw_chi.map(custom_mapping)
y_test_chi = y_test_raw_chi.map(custom_mapping)

print(f"      Jumlah Dimensi Fitur: {X_train_chi.shape[1]}")
print("      Sedang melatih Baseline Model (Naive Bayes)...")

# --- INISIALISASI & TRAINING MODEL ---
model_baseline = MultinomialNB()
model_baseline.fit(X_train_chi, y_train_chi)

# --- EVALUASI EKSPERIMEN 2 ---
y_pred_nb = model_baseline.predict(X_test_chi)
acc_nb = accuracy_score(y_test_chi, y_pred_nb)

print("\nHASIL EKSPERIMEN 2 (BASELINE - NAIVE BAYES):")
print("-" * 60)
print(f"Akurasi Keseluruhan: {acc_nb:.2%}")
print("\nClassification Report:")
print(classification_report(y_test_chi, y_pred_nb, target_names=target_names))
print("="*60)

# --- REKAPITULASI HASIL UNTUK DOKUMENTASI ---
print("\nREKAPITULASI HASIL UNTUK TABEL BAB 4:")
print("-" * 60)
print(f"1. Model Original (LogReg + Chi2) : ... % (Mohon periksa data pengujian utama)")
print(f"2. Tanpa Chi-Square (LogReg Full) : {acc_full:.2%}")
print(f"3. Baseline Model (Naive Bayes)   : {acc_nb:.2%}")
print("="*60)